In [ ]:
# !pip install jiwer
# !pip install faster-whisper # whisper library
# !pip install deepgram-sdk # deepgram library
# !pip install elevenlabs # elevenlabs library
# !pip install httpx
# !pip install torchaudio
# !pip install librosa
# !pip install datasets
# !pip install --upgrade datasets #-> datasets update if there's an error
# !pip install psutil
# !pip install torchaudio
# !pip install typing

# # Tracking
# #################################
# !pip install mlflow # tracking
# !pip install pyngrok # ngrok
# ##############################

In [ ]:
import subprocess
import mlflow # mlFlow
from pyngrok import ngrok # workaround for localhost

wandb_token = "write_your_token_here"
ngrok_token = "2zMMNLcnHN70HISWDBeIt66RoJb_BuLumxNNBiGAeRjo9Dha" # optional, mlFlow

# https://dashboard.ngrok.com/authtokens
ngrok.set_auth_token(ngrok_token)
port = "5000"

mlflow_proc = subprocess.Popen(["mlflow", "ui", "--port", port])
mlflow.autolog()
# mlflow_proc.terminate()

public_url = ngrok.connect(port)
print(f"MLflow UI: {public_url}")

In [ ]:
############################
# Whisper dependencies
from faster_whisper import WhisperModel
############################

# Getting file from colab
# from google.colab import files
# uploaded = files.upload()
# filename = next(iter(uploaded))

###############################
# DeepGram dependencies
import requests
import wave
import io
import time
import os
import torch
import logging
import json
import threading
from datetime import datetime
import deepgram
from deepgram import (
  DeepgramClient,
  DeepgramClientOptions,
  AgentWebSocketEvents,
  AgentKeepAlive,
  PrerecordedOptions,
  FileSource
)
################################

# Elevenlabs dependencies
################################
from elevenlabs.client import ElevenLabs
import os
from io import BytesIO
import requests
################################

# Initialize the client

"""
MODELS
  Whisper:
    1. tiny -> ~ 39 millions
    2. medium -> ~ 769 millions
    3. large-v1 -> ~1.55 billions
    4. turbo -> ~1.55 billions, optimized version of large-v1
    5. distill-large-v3 -> ~400-500 millions, lighter than large-v3
    6. large-v3 ->  ~ 1.6 billion parameters
  Deepgram:
    1. nova2 -> official and only model which support greek language by default
  HugginFace:
    1. jonatasgrosman/wav2vec2-large-xlsr-53-greek -> facebook wav2vec2 finetuned greek model, ~ 317 billion parameters, 166k downloads
    Note: the only useful custom hf model, others are 1k downloads on avg, not serious
  Elevenlabs:
    1. Scribe-1 -> probably the newest model on a market
"""

# Notes:
# 1. ElevenLabs can't be used, cause it has text-to-speech format only
# 2. microsoft/wavlm-large # does not work, because it works with embeddings, not text
# 3. facebook/wav2vec2-large-xlsr-53 -> facebook normal model, not finetuned, does not work , because it does not have trained weights, need a specific one

# WHISPER Models
# whisper_tiny = WhisperModel("tiny")
# whisper_medium = WhisperModel("medium")
# whisper_large_v1 = WhisperModel("large-v1")
# whisper_turbo = WhisperModel("turbo")
# whisper_distill_large_v3 = WhisperModel("distill-large-v3")
# whisper_large_v3 = WhisperModel("large-v3")

###################################
# Mozilla Common Voice for testing
from datasets import load_dataset, Audio

# Greek dataset
dataset = load_dataset("mozilla-foundation/common_voice_17_0", "el", split="train")

# Squeezing to 16hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
###################################

In [56]:
"""
Model Loader Pipeline
"""

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

import torch.nn as nn
import torch
import torchaudio
import soundfile as sf

class ModelPipeline:
  def __init__(self, model_name, audio):
    self.model_name = model_name
    self.audio = audio
    self.processor = None
    self.model = None

  # Whisper
  #####################################
  def load_whisper_model(self):
    self.model = WhisperModel(self.model_name)

  def whisper_process_logic(self):
    text_stored = ""
    segments, info = self.model.transcribe(self.audio, language="el") # <- specify language output
    for segment in segments:
      # print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text)) # <- to see with seconds
      text_stored += segment.text + " "
    return text_stored
  #####################################

  # HuggingFace
  #####################################
  def load_hf_model(self):
    self.model = Wav2Vec2ForCTC.from_pretrained(self.model_name)

  def load_hf_processor(self):
    self.processor = Wav2Vec2Processor.from_pretrained(self.model_name)

  def hf_process_logic(self):
    speech = self.audio["array"]
    sr = self.audio["sampling_rate"]

    if sr != 16000:
        import librosa
        speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)
        sr = 16000

    inputs = self.processor(speech, sampling_rate=sr, return_tensors="pt")

    with torch.no_grad():
        logits = self.model(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = self.processor.decode(predicted_ids[0])

    return transcription
  #####################################

  # DeepGram
  #####################################
  def deepgram_process_logic(self):
        # Handle different audio input formats
        if isinstance(self.audio, dict) and "array" in self.audio:
            # HuggingFace dataset format
            audio_array = self.audio["array"]
            sample_rate = self.audio["sampling_rate"]
        elif isinstance(self.audio, str):
            # File path format
            audio_array, sample_rate = sf.read(self.audio)
        else:
            raise ValueError(f"Unsupported audio format: {type(self.audio)}")

        # Convert to bytes buffer
        buffer = io.BytesIO()
        sf.write(buffer, audio_array, sample_rate, format='WAV')
        buffer.seek(0)  # Reset to the beginning

        # Configure options
        options = PrerecordedOptions(
            model="nova-2",
            smart_format=True,
            language="el",
        )

        # Transcribe using buffer data
        response = deepgram.listen.rest.v("1").transcribe_file(
            {"buffer": buffer, "mimetype": "audio/wav"},
            options
        )
        transcript = response["results"]["channels"][0]["alternatives"][0]["transcript"]

        return transcript
  #####################################

  # Elevenlabs
  #####################################
  def eleven_process_logic(self):
    # Check if self.audio is a URL or a local file path
    if self.audio.startswith(('http://', 'https://')):
        # It's a URL, use requests to download
        response = requests.get(self.audio)
        response.raise_for_status()  # Raise an exception for bad status codes
        audio_data = BytesIO(response.content)
    else:
        # It's a local file path, read directly
        try:
            with open(self.audio, 'rb') as f:
                audio_data = BytesIO(f.read())
        except FileNotFoundError:
            raise FileNotFoundError(f"Audio file not found: {self.audio}")
        except PermissionError:
            raise PermissionError(f"Permission denied accessing audio file: {self.audio}")

    transcription = elevenlabs_client.speech_to_text.convert(
        file=audio_data,
        model_id="scribe_v1", # Model to use, for now only "scribe_v1" is supported
        tag_audio_events=True, # Tag audio events like laughter, applause, etc.
        language_code="ell", # Language of the audio file. If set to None, the model will detect the language automatically.
        diarize=True, # Whether to annotate who is speaking
    )
    return transcription.text # <- there was an error with output
  #####################################

In [ ]:
"""
  Jiwer Loader Pipeline
"""
import jiwer
from jiwer import wer,cer

"""
wer - word error rate
cer - character error rate
"""

"""

ValueError: After applying the transformation, each reference should be a list of strings, with each string being a single word or character.Please ensure the given transformation reduces the input to a list of list strings.

To fix:

jiwer.ToLowerCase(),
jiwer.RemovePunctuation(),
jiwer.RemoveMultipleSpaces(),
jiwer.Strip()

and adding a normalization
"""

class JiwerMetricsPipeline:
  def __init__(self, hypothesis, truth):
    self.hypothesis = hypothesis
    self.truth = truth

  def compute_metrics(self):
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip()
    ])

    normalized_truth = transformation(self.truth)
    normalized_hypothesis = transformation(self.hypothesis)

    wer_score = wer(normalized_truth, normalized_hypothesis)
    cer_score = cer(normalized_truth, normalized_hypothesis)

    return wer_score, cer_score

In [ ]:
stored = []

In [ ]:
import psutil
import pandas as pd
import numpy as np

run_counter = 0

# Metrics extensions
from mlflow.models import infer_signature
from mlflow.data.pandas_dataset import PandasDataset

def launch_model(model, stored, sentence, audio):
  global run_counter
  run_counter += 1
  with mlflow.start_run(run_name=f"{model}_audio_{run_counter}"):

    # Dataset Mlflow
    #################################
    dataset_info = pd.DataFrame({
      'audio_path': [audio],
      'truth_text': [sentence],
      'text_length': [len(sentence)]
    })

    dataset = mlflow.data.from_pandas(
      dataset_info,
      source=f"huggingface",
      name="mozilla_common_voice",
    )

    mlflow.log_input(dataset, context="training")
    #################################

    # Log params for mlflow
    #################################
    mlflow.log_param("model_name", model)
    # mlflow.log_param("audio_file", os.path.basename(audio))
    mlflow.log_param("timestamp", datetime.now().isoformat())
    #################################

    # Specs logging
    #################################
    mlflow.log_param("cpu_count", psutil.cpu_count())
    mlflow.log_param("memory_total_gb", round(psutil.virtual_memory().total / 1024**3, 1))
    #################################

    # Time logging
    ##########################
    start_time = time.time()

    llm = ModelPipeline(model, audio)

    memory_before = psutil.virtual_memory().used / 1024**3 # before whisper.process_logic

    """
    whisper
    """
    # llm.load_whisper_model()

    # process_start = time.time()
    # whisper_output = llm.whisper_process_logic()
    # process_time = time.time() - process_start

    """
    huggingface
    """
    # llm.load_hf_model()
    # llm.load_hf_processor()

    # process_start = time.time()
    # hf_output = llm.hf_process_logic()
    # process_time = time.time() - process_start

    """
    deepgram
    """
    # process_start = time.time()
    # deepgram_output = llm.deepgram_process_logic()
    # process_time = time.time() - process_start

    """
    elevenlabs
    """
    process_start = time.time()
    deepgram_output = llm.eleven_process_logic()
    process_time = time.time() - process_start

    total_time = time.time() - start_time

    memory_after = psutil.virtual_memory().used / 1024**3 # after whisper.process_logic

    mlflow.log_metric("process_time_seconds", round(process_time,2))
    mlflow.log_metric("total_time_seconds", total_time)
    mlflow.log_metric("memory_usage_change_gb", round(memory_after - memory_before, 3))
    mlflow.log_metric("cpu_usage_percent", psutil.cpu_percent())
    #############################

    """
    Model logging does not work, because it must be working on a local server, not a cloud, seems it does not accept json format also
    """

    # Model logging
    # #################################
    # input_model = {
    #   "audio_path": audio,
    #   "model_name": model
    # }

    # output_model = {
    #   "transcription": whisper_output,
    #   "processing_time": process_time
    # }

    # signature = infer_signature(input_model, output_model)

    # model_info = {
    #   "model_name": model,
    #   "model_type": "whisper",
    #   "framework": "openai-whisper",
    #   "version": "1.0",
    #   "parameters": {
    #       "model_size": model,
    #       "language": "auto"
    #   }
    # }

    # mlflow.log_dict(model_info, "model_info.json")
    # #################################

    err_metrics = JiwerMetricsPipeline(deepgram_output, sentence).compute_metrics()

    stored.append({
      "index": len(stored),
      "model": model,
      "wer": round(err_metrics[0], 2),
      "cer": round(err_metrics[1], 2)
    })

    #wer, cer
    mlflow.log_metric("total_wer", round(err_metrics[0], 2))
    mlflow.log_metric("total_cer", round(err_metrics[1], 2))

    # mlflow.log_dict(stored[-1], f"wer_cer_data.json")

    return stored

In [59]:
# Common launch
for i in range(10):

    dataset_sentence = dataset[i]["sentence"]
    # for whisper models
    path_audio = dataset[i]["audio"]["path"]
    # for hf models and deepgram models
    common_audio = dataset[i]["audio"]

    launch_model("elevenlabs", stored, dataset_sentence, path_audio)

    print(f"Processed sample {i+1}/10")

/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'huggingface'. Exception: 
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data samp

Processed sample 1/10


/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'huggingface'. Exception: 
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data samp

Processed sample 2/10


/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'huggingface'. Exception: 
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data samp

...

Processed sample 8/10


/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'huggingface'. Exception: 
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data samp

Processed sample 9/10


/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'huggingface'. Exception: 
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/data/dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data samp

Processed sample 10/10
